In [1]:
import pandas as pd
import numpy as np
import sqlite3
import datetime

# ==========================================
# LAYER 1: TELEMETRY DATA SIMULATION (Big Data Ingestion)
# ==========================================
def generate_clickstream_telemetry(n_sessions=25000):
    """Simulates raw event logs of users navigating the digital risk journey."""
    np.random.seed(42)
    events = []
    base_time = datetime.datetime.now()

    for i in range(n_sessions):
        sess_id = f"TXN-{100000+i}"
        # A/B Testing: 50% get the old legacy flow, 50% get the new Optimized API flow
        variant = "Control_Legacy" if np.random.rand() > 0.5 else "Variant_Optimized_API"

        # Step 1: User starts the application
        events.append((sess_id, variant, '1_Landing_Page', base_time))

        # Step 2: Income Verification (Friction Point)
        drop_rate = 0.48 if variant == "Control_Legacy" else 0.12 # Legacy loses 48% of users here
        if np.random.rand() > drop_rate:
            events.append((sess_id, variant, '2_Income_Upload', base_time + datetime.timedelta(seconds=np.random.randint(15, 120))))

            # Step 3: Final Decision Received
            if np.random.rand() > 0.05: # 5% system timeout failure
                events.append((sess_id, variant, '3_Decision_Received', base_time + datetime.timedelta(seconds=np.random.randint(150, 200))))

    return pd.DataFrame(events, columns=['session_id', 'journey_variant', 'event_name', 'event_timestamp'])

telemetry_df = generate_clickstream_telemetry()

# ==========================================
# LAYER 2: IN-MEMORY SQL DATABASE DEPLOYMENT
# ==========================================
# Create a local SQL engine to demonstrate BigQuery/SQL skills natively in Python
conn = sqlite3.connect(':memory:')
telemetry_df.to_sql('user_telemetry', conn, index=False)

# ==========================================
# LAYER 3: ADVANCED SQL FUNNEL ANALYTICS
# ==========================================
# Using Common Table Expressions (CTEs) and Conditional Aggregation
sql_query = """
WITH Funnel_Aggregation AS (
    SELECT
        journey_variant,
        COUNT(DISTINCT CASE WHEN event_name = '1_Landing_Page' THEN session_id END) AS total_starts,
        COUNT(DISTINCT CASE WHEN event_name = '2_Income_Upload' THEN session_id END) AS successful_uploads,
        COUNT(DISTINCT CASE WHEN event_name = '3_Decision_Received' THEN session_id END) AS completed_decisions
    FROM user_telemetry
    GROUP BY journey_variant
)
SELECT
    journey_variant,
    total_starts,
    successful_uploads,
    ROUND((successful_uploads * 100.0 / total_starts), 2) AS upload_conversion_pct,
    completed_decisions,
    ROUND((completed_decisions * 100.0 / total_starts), 2) AS end_to_end_conversion_pct
FROM Funnel_Aggregation;
"""

funnel_report = pd.read_sql_query(sql_query, conn)

# ==========================================
# LAYER 4: STRATEGIC BUSINESS INSIGHT OUTPUT
# ==========================================
print("=== DIGITAL JOURNEY FRICTION & A/B TEST REPORT ===")
print(funnel_report.to_string(index=False))

print("\n=== DATA STRATEGY RECOMMENDATION ===")
legacy_conv = funnel_report.loc[funnel_report['journey_variant'] == 'Control_Legacy', 'end_to_end_conversion_pct'].values[0]
api_conv = funnel_report.loc[funnel_report['journey_variant'] == 'Variant_Optimized_API', 'end_to_end_conversion_pct'].values[0]
lift = api_conv - legacy_conv

print(f"Insight: The 'Variant_Optimized_API' flow increased end-to-end customer conversion by {lift:.2f}%.")
print("Actionable Recommendation: Deprecate the legacy income upload UI. Full migration to the Optimized API will reduce decision friction and recapture lost customer acquisition.")

=== DIGITAL JOURNEY FRICTION & A/B TEST REPORT ===
      journey_variant  total_starts  successful_uploads  upload_conversion_pct  completed_decisions  end_to_end_conversion_pct
       Control_Legacy         12464                6537                  52.45                 6209                      49.82
Variant_Optimized_API         12536               11004                  87.78                10410                      83.04

=== DATA STRATEGY RECOMMENDATION ===
Insight: The 'Variant_Optimized_API' flow increased end-to-end customer conversion by 33.22%.
Actionable Recommendation: Deprecate the legacy income upload UI. Full migration to the Optimized API will reduce decision friction and recapture lost customer acquisition.
